In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                           balanced_accuracy_score, confusion_matrix, classification_report,
                           roc_auc_score, roc_curve, precision_recall_curve, auc,
                           matthews_corrcoef, cohen_kappa_score, log_loss)
from sklearn.preprocessing import label_binarize
from sklearn.calibration import calibration_curve
import time
import os
from itertools import cycle
from scipy import stats

# Configuration parameters
K_FOLDS = 5
RANDOM_STATE = 42
OUTPUT_DIRS = ["confusion_matrices", "fld_plots", "roc_curves", "pr_curves", "calibration_plots"]

# Create output directories
for directory in OUTPUT_DIRS:
    os.makedirs(directory, exist_ok=True)

def load_and_preprocess_data(file_path):
    """Load data from CSV and perform initial preprocessing."""
    df = pd.read_csv(file_path)
    
    print(f"Dataset shape: {df.shape}")
    print(f"Number of classes: {df['label'].nunique()}")
    print(f"Class distribution:\n{df['label'].value_counts()}")
    
    # Check for missing values
    missing_values = df.isnull().sum().sum()
    print(f"\nMissing values in the dataset: {missing_values}")
    
    # Handle missing values if any
    if missing_values > 0:
        df = df.fillna(0)
    
    # Extract features and target
    X = df.drop(columns=['label']).values
    y = df['label'].values
    
    return X, y, df['label'].unique()

def confusion_matrix_metrics(cm, classes):
    """Calculate detailed metrics from a confusion matrix."""
    metrics = {}
    for idx, class_label in enumerate(classes):
        TP = cm[idx, idx]
        FP = cm[:, idx].sum() - TP
        FN = cm[idx, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)
 
        # Avoid division by zero with epsilon
        eps = 1e-10
        TPR = TP / (TP + FN + eps) if (TP + FN) > 0 else 0  # Sensitivity/Recall
        TNR = TN / (TN + FP + eps) if (TN + FP) > 0 else 0  # Specificity
        FPR = FP / (FP + TN + eps) if (FP + TN) > 0 else 0  # Fall-out
        FNR = FN / (FN + TP + eps) if (FN + TP) > 0 else 0  # Miss rate
        PPV = TP / (TP + FP + eps) if (TP + FP) > 0 else 0  # Precision
        NPV = TN / (TN + FN + eps) if (TN + FN) > 0 else 0  # Negative Predictive Value
        FDR = FP / (TP + FP + eps) if (TP + FP) > 0 else 0  # False Discovery Rate
        FOR = FN / (TN + FN + eps) if (TN + FN) > 0 else 0  # False Omission Rate
        
        # Calculate more complex metrics
        LR_plus = TPR / (FPR + eps)  # Positive Likelihood Ratio
        LR_minus = FNR / (TNR + eps)  # Negative Likelihood Ratio
        DOR = (TP * TN) / ((FP * FN) + eps)  # Diagnostic Odds Ratio
        F1 = 2 * TP / (2 * TP + FP + FN + eps)  # F1 Score
        ACC = (TP + TN) / (TP + TN + FP + FN + eps)  # Accuracy
        BA = (TPR + TNR) / 2  # Balanced Accuracy
        
        # Matthews Correlation Coefficient
        MCC_numerator = (TP * TN - FP * FN)
        MCC_denominator = np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN) + eps)
        MCC = MCC_numerator / MCC_denominator
        
        metrics[class_label] = {
            'TPR': TPR, 'TNR': TNR, 'FPR': FPR, 'FNR': FNR,
            'Precision': PPV, 'NPV': NPV, 'FDR': FDR, 'FOR': FOR,
            'LR+': LR_plus, 'LR-': LR_minus, 'DOR': DOR,
            'F1': F1, 'Accuracy': ACC, 'Balanced Accuracy': BA, 'MCC': MCC
        }
    
    return metrics

def perform_fld(X_train, y_train, X_test, n_components=None):
    """Perform Fisher's Linear Discriminant analysis."""
    # If n_components is not specified, use min(n_features, n_classes-1)
    if n_components is None:
        n_components = min(X_train.shape[1], len(np.unique(y_train)) - 1)
    
    # Initialize LDA
    lda = LinearDiscriminantAnalysis(n_components=n_components)
    
    # Fit and transform the training data
    X_train_lda = lda.fit_transform(X_train, y_train)
    
    # Transform the test data
    X_test_lda = lda.transform(X_test)
    
    return lda, X_train_lda, X_test_lda
def plot_fld_projection(X_train_lda, y_train, clf_name, fold_idx):
    """Plot the FLD projections if dimensionality allows."""
    if X_train_lda.shape[1] < 2:
        return  # Skip if not enough dimensions for 2D plot
    
    plt.figure(figsize=(10, 8))
    unique_classes = np.unique(y_train)
    
    for class_label in unique_classes:
        class_mask = (y_train == class_label)
        plt.scatter(X_train_lda[class_mask, 0], X_train_lda[class_mask, 1], 
                   label=f'Class {class_label}', alpha=0.7)
    
    plt.title(f'FLD Projection - {clf_name} - Fold {fold_idx}')
    plt.xlabel('LD1')
    plt.ylabel('LD2')
    plt.legend(loc='best')
    plt.tight_layout()
    plt.savefig(f"fld_plots/fld_projection_{clf_name}_fold_{fold_idx}.png")
    plt.close()

def plot_confusion_matrix(cm, unique_classes, clf_name, fold_idx):
    """Plot and save confusion matrix."""
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
               xticklabels=unique_classes, yticklabels=unique_classes)
    plt.title(f"{clf_name} - Fold {fold_idx} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(f"confusion_matrices/{clf_name}_fold_{fold_idx}.png")
    plt.close()

def plot_roc_curve(y_test, y_scores, classes, clf_name, fold_idx):
    """Plot and save ROC curve."""
    plt.figure(figsize=(10, 8))
    
    # Binary classification case
    if len(classes) == 2:
        fpr, tpr, _ = roc_curve(y_test, y_scores[:, 1])
        roc_auc = auc(fpr, tpr)
        
        plt.plot(fpr, tpr, lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--', lw=2)
    
    # Multiclass classification case
    else:
        # Binarize the output
        y_test_bin = label_binarize(y_test, classes=classes)
        n_classes = y_test_bin.shape[1]
        
        # Compute ROC curve and ROC area for each class
        fpr = dict()
        tpr = dict()
        roc_auc = dict()
        
        colors = cycle(['blue', 'red', 'green', 'yellow', 'purple', 'orange', 'cyan'])
        
        for i, color in zip(range(n_classes), colors):
            try:
                fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_scores[:, i])
                roc_auc[i] = auc(fpr[i], tpr[i])
                plt.plot(fpr[i], tpr[i], color=color, lw=2,
                        label=f'ROC class {classes[i]} (AUC = {roc_auc[i]:.2f})')
            except (ValueError, IndexError):
                continue
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{clf_name} - ROC Curve (Fold {fold_idx})')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"roc_curves/{clf_name}_fold_{fold_idx}_roc.png")
    plt.close()

def plot_pr_curve(y_test, y_scores, classes, clf_name, fold_idx):
    """Plot and save Precision-Recall curve."""
    plt.figure(figsize=(10, 8))
    
    # Binary classification case
    if len(classes) == 2:
        precision, recall, _ = precision_recall_curve(y_test, y_scores[:, 1])
        pr_auc = auc(recall, precision)
        
        plt.plot(recall, precision, lw=2, label=f'PR curve (area = {pr_auc:.2f})')
    
    # Multiclass classification case
    else:
        # Binarize the output
        y_test_bin = label_binarize(y_test, classes=classes)
        n_classes = y_test_bin.shape[1]
        
        # Compute Precision-Recall curve and area for each class
        precision = dict()
        recall = dict()
        pr_auc = dict()
        
        colors = cycle(['blue', 'red', 'green', 'yellow', 'purple', 'orange', 'cyan'])
        
        for i, color in zip(range(n_classes), colors):
            try:
                precision[i], recall[i], _ = precision_recall_curve(y_test_bin[:, i], y_scores[:, i])
                pr_auc[i] = auc(recall[i], precision[i])
                plt.plot(recall[i], precision[i], color=color, lw=2,
                        label=f'PR class {classes[i]} (AUC = {pr_auc[i]:.2f})')
            except (ValueError, IndexError):
                continue
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'{clf_name} - Precision-Recall Curve (Fold {fold_idx})')
    plt.legend(loc="best")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"pr_curves/{clf_name}_fold_{fold_idx}_pr.png")
    plt.close()

def plot_calibration_curve(y_test, y_proba, clf_name, fold_idx):
    """Plot and save calibration curve."""
    plt.figure(figsize=(10, 8))
    
    # Plot calibration curve for each class
    for i in range(y_proba.shape[1]):
        try:
            prob_true, prob_pred = calibration_curve(y_test == i, y_proba[:, i], n_bins=10)
            plt.plot(prob_pred, prob_true, marker='o', linewidth=1, label=f'Class {i}')
        except (ValueError, IndexError):
            continue
    
    plt.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
    plt.xlabel('Mean predicted probability')
    plt.ylabel('Fraction of positives')
    plt.title(f'{clf_name} - Calibration Curve (Fold {fold_idx})')
    plt.legend(loc="best")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"calibration_plots/{clf_name}_fold_{fold_idx}_calibration.png")
    plt.close()

def calculate_class_metrics(y_test, y_pred, class_label, cm_metrics, fold_idx, clf_name, timing_info):
    """Calculate metrics for a specific class."""
    class_mask = (y_test == class_label)
    
    # Skip classes with no instances in the test set
    if class_mask.sum() == 0:
        return {
            'Classifier': clf_name,
            'Fold': fold_idx,
            'Class': class_label,
            'Accuracy': np.nan,
            'Precision': np.nan,
            'Recall': np.nan,
            'F1 Score': np.nan,
            'Balanced Accuracy': np.nan,
            'MCC': np.nan,
            'True Positive Rate (TPR)': np.nan,
            'True Negative Rate (TNR)': np.nan,
            'False Positive Rate (FPR)': np.nan,
            'False Negative Rate (FNR)': np.nan,
            'FLD Time (s)': timing_info.get('fld_time', 0),
            'Training Time (s)': timing_info.get('train_time', 0),
            'Testing Time (s)': timing_info.get('test_time', 0),
            'Total Time (s)': sum(timing_info.values())
        }
    
    # Calculate metrics for this class
    binary_y_test = (y_test == class_label).astype(int)
    binary_y_pred = (y_pred == class_label).astype(int)
    
    metrics = {
        'Classifier': clf_name,
        'Fold': fold_idx,
        'Class': class_label,
        'Accuracy': accuracy_score(binary_y_test, binary_y_pred),
        'Precision': precision_score(binary_y_test, binary_y_pred, zero_division=0),
        'Recall': recall_score(binary_y_test, binary_y_pred, zero_division=0),
        'F1 Score': f1_score(binary_y_test, binary_y_pred, zero_division=0),
        'Balanced Accuracy': balanced_accuracy_score(binary_y_test, binary_y_pred),
        'MCC': matthews_corrcoef(binary_y_test, binary_y_pred),
        'FLD Time (s)': timing_info.get('fld_time', 0),
        'Training Time (s)': timing_info.get('train_time', 0),
        'Testing Time (s)': timing_info.get('test_time', 0),
        'Total Time (s)': sum(timing_info.values())
    }
    
    # Add confusion matrix derived metrics
    if class_label in cm_metrics:
        for metric_name, value in cm_metrics[class_label].items():
            metrics[metric_name] = value
    
    return metrics

def calculate_log_loss_safely(y_test, y_proba, classes):
    """Calculate log loss with error handling."""
    try:
        return log_loss(y_test, y_proba, labels=classes)
    except (ValueError, IndexError) as e:
        print(f"Warning: Log loss calculation failed: {e}")
        return np.nan

def calculate_roc_auc_safely(y_test, y_proba):
    """Calculate ROC AUC score with error handling."""
    try:
        return roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
    except (ValueError, IndexError) as e:
        print(f"Warning: ROC AUC calculation failed: {e}")
        return np.nan

def save_classification_report(y_test, y_pred, clf_name, fold_idx):
    """Generate and save classification report."""
    report = classification_report(y_test, y_pred, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(f"classification_report_{clf_name}_fold_{fold_idx}.csv")

def generate_summary_visualizations(all_metrics_df, results_df, timing_df):
    """Generate summary visualizations from the results."""
    # Accuracy by classifier boxplot
    plt.figure(figsize=(12, 8))
    sns.boxplot(x='Classifier', y='Accuracy', data=all_metrics_df)
    plt.title('Accuracy by Classifier')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('fld_accuracy_comparison.png')
    plt.close()
    
    # F1 score by classifier and class
    plt.figure(figsize=(14, 10))
    sns.boxplot(x='Classifier', y='F1 Score', hue='Class', data=results_df)
    plt.title('F1 Score by Classifier and Class')
    plt.xticks(rotation=45)
    plt.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('fld_f1_comparison_by_class.png')
    plt.close()
    
    # MCC by classifier
    plt.figure(figsize=(12, 8))
    sns.boxplot(x='Classifier', y='MCC', data=all_metrics_df)
    plt.title('Matthews Correlation Coefficient by Classifier')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('fld_mcc_comparison.png')
    plt.close()
    
    # ROC AUC by classifier
    plt.figure(figsize=(12, 8))
    sns.boxplot(x='Classifier', y='ROC AUC', data=all_metrics_df)
    plt.title('ROC AUC by Classifier')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('fld_roc_auc_comparison.png')
    plt.close()
    
    # Training time by classifier
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Classifier', y='Training Time (s)', 
               data=timing_df.groupby('Classifier')['Training Time (s)'].mean().reset_index())
    plt.title('Average Training Time by Classifier')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('fld_training_time_comparison.png')
    plt.close()
    
    # Total execution time by classifier
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Classifier', y='Total Time (s)', 
               data=timing_df.groupby('Classifier')['Total Time (s)'].mean().reset_index())
    plt.title('Average Total Execution Time by Classifier')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('fld_total_time_comparison.png')
    plt.close()
    
    # Correlation between metrics
    metric_columns = ['Accuracy', 'Balanced Accuracy', 'MCC', 'Kappa', 'ROC AUC', 'Log Loss']
    plt.figure(figsize=(12, 10))
    correlation = all_metrics_df[metric_columns].corr()
    sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Between Metrics')
    plt.tight_layout()
    plt.savefig('fld_metrics_correlation.png')
    plt.close()
    
    # Create radar chart
    create_radar_chart(all_metrics_df)

def create_radar_chart(all_metrics_df):
    """Create a radar chart to compare classifiers across multiple metrics."""
    # Prepare metrics for radar chart
    radar_metrics = ['Accuracy', 'Balanced Accuracy', 'MCC', 'Kappa', 'ROC AUC']
    
    # Normalize Log Loss to [0,1] range for radar chart if needed
    df_mean = all_metrics_df.groupby('Classifier')[radar_metrics].mean().reset_index()
    
    # Number of variables
    categories = radar_metrics
    N = len(categories)
    
    # Create angles for each metric
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]  # Close the loop
    
    # Create the plot
    fig = plt.figure(figsize=(10, 10))
    ax = plt.subplot(111, polar=True)
    
    # Draw one axis per variable and add labels
    plt.xticks(angles[:-1], categories, size=12)
    
    # Draw the y-axis labels (0 to 1)
    ax.set_rlabel_position(0)
    plt.yticks([0.2, 0.4, 0.6, 0.8], ["0.2", "0.4", "0.6", "0.8"], size=10)
    plt.ylim(0, 1)
    
    # Plot each classifier
    for i, classifier in enumerate(df_mean['Classifier']):
        values = df_mean.loc[i, radar_metrics].values.flatten().tolist()
        values += values[:1]  # Close the loop
        
        # Plot values
        ax.plot(angles, values, linewidth=2, linestyle='solid', label=classifier)
        ax.fill(angles, values, alpha=0.1)
    
    # Add legend
    plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
    plt.title('Classifier Performance Comparison', size=15, y=1.1)
    
    plt.savefig('fld_radar_comparison.png', bbox_inches='tight')
    plt.close(fig)

def perform_statistical_analysis(all_metrics_df):
    """Perform statistical analysis on the results."""
    # Calculate statistical significance between classifiers
    classifiers_list = all_metrics_df['Classifier'].unique()
    
    # Create a DataFrame to store statistical test results
    stat_results = []
    
    # Compare accuracy between classifiers
    for i, clf1 in enumerate(classifiers_list):
        for j, clf2 in enumerate(classifiers_list):
            if i < j:  # Avoid duplicate comparisons
                group1 = all_metrics_df[all_metrics_df['Classifier'] == clf1]['Accuracy']
                group2 = all_metrics_df[all_metrics_df['Classifier'] == clf2]['Accuracy']
                
                # Perform t-test
                t_stat, p_val = stats.ttest_ind(group1, group2, equal_var=False)
                
                stat_results.append({
                    'Classifier1': clf1,
                    'Classifier2': clf2,
                    'Metric': 'Accuracy',
                    't-statistic': t_stat,
                    'p-value': p_val,
                    'Significant': p_val < 0.05
                })
    
    # Create DataFrame with statistical test results
    stat_df = pd.DataFrame(stat_results)
    stat_df.to_csv("fld_statistical_tests.csv", index=False)
    
    # Create a heatmap of p-values
    plt.figure(figsize=(10, 8))
    heatmap_data = np.zeros((len(classifiers_list), len(classifiers_list)))
    
    for i, clf1 in enumerate(classifiers_list):
        for j, clf2 in enumerate(classifiers_list):
            if i < j:
                p_val = stat_df[(stat_df['Classifier1'] == clf1) & 
                               (stat_df['Classifier2'] == clf2)]['p-value'].values[0]
                heatmap_data[i, j] = p_val
                heatmap_data[j, i] = p_val  # Mirror for complete heatmap
            elif i == j:
                heatmap_data[i, j] = 1.0  # p-value of 1 for same classifier
    
    # Plot heatmap
    mask = np.triu(np.ones_like(heatmap_data, dtype=bool), k=1)
    sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="YlGnBu", 
               xticklabels=classifiers_list, yticklabels=classifiers_list,
               mask=mask)
    plt.title('p-values from t-tests comparing Accuracy between Classifiers')
    plt.tight_layout()
    plt.savefig('fld_pvalue_heatmap.png')
    plt.close()
    
    return stat_df

def print_final_report(all_metrics_df, timing_df, results_df):
    """Print a final report with summary statistics."""
    # Compute mean metrics by classifier
    summary = all_metrics_df.groupby('Classifier').agg({
        'Accuracy': ['mean', 'std'],
        'Balanced Accuracy': ['mean', 'std'],
        'MCC': ['mean', 'std'],
        'Kappa': ['mean', 'std'],
        'ROC AUC': ['mean', 'std'],
        'Log Loss': ['mean', 'std'],
        'Total Time (s)': ['mean', 'std']
    }).reset_index()
    
    # Find best classifier for each metric
    best_accuracy = summary.loc[summary[('Accuracy', 'mean')].idxmax()]
    best_balanced_acc = summary.loc[summary[('Balanced Accuracy', 'mean')].idxmax()]
    best_mcc = summary.loc[summary[('MCC', 'mean')].idxmax()]
    best_roc_auc = summary.loc[summary[('ROC AUC', 'mean')].idxmax()]
    fastest = summary.loc[summary[('Total Time (s)', 'mean')].idxmin()]
    
    # Print summary report
    print("\n" + "="*80)
    print("FISHER'S LINEAR DISCRIMINANT ANALYSIS EVALUATION REPORT")
    print("="*80)
    
    print("\nBest classifier by Accuracy: {} ({:.4f} ± {:.4f})".format(
        best_accuracy['Classifier'], 
        best_accuracy[('Accuracy', 'mean')], 
        best_accuracy[('Accuracy', 'std')]
    ))
    
    print("Best classifier by Balanced Accuracy: {} ({:.4f} ± {:.4f})".format(
        best_balanced_acc['Classifier'], 
        best_balanced_acc[('Balanced Accuracy', 'mean')], 
        best_balanced_acc[('Balanced Accuracy', 'std')]
    ))
    
    print("Best classifier by MCC: {} ({:.4f} ± {:.4f})".format(
        best_mcc['Classifier'], 
        best_mcc[('MCC', 'mean')], 
        best_mcc[('MCC', 'std')]
    ))
    
    print("Best classifier by ROC AUC: {} ({:.4f} ± {:.4f})".format(
        best_roc_auc['Classifier'], 
        best_roc_auc[('ROC AUC', 'mean')], 
        best_roc_auc[('ROC AUC', 'std')]
    ))
    
    print("Fastest classifier: {} ({:.4f} ± {:.4f} seconds)".format(
        fastest['Classifier'], 
        fastest[('Total Time (s)', 'mean')], 
        fastest[('Total Time (s)', 'std')]
    ))
    
    print("\nComplete performance summary:")
    for _, row in summary.iterrows():
        clf = row['Classifier']
        print("\n{} Performance:".format(clf))
        print("  Accuracy: {:.4f} ± {:.4f}".format(row[('Accuracy', 'mean')], row[('Accuracy', 'std')]))
        print("  Balanced Accuracy: {:.4f} ± {:.4f}".format(row[('Balanced Accuracy', 'mean')], row[('Balanced Accuracy', 'std')]))
        print("  MCC: {:.4f} ± {:.4f}".format(row[('MCC', 'mean')], row[('MCC', 'std')]))
        print("  Kappa: {:.4f} ± {:.4f}".format(row[('Kappa', 'mean')], row[('Kappa', 'std')]))
        print("  ROC AUC: {:.4f} ± {:.4f}".format(row[('ROC AUC', 'mean')], row[('ROC AUC', 'std')]))
        if not np.isnan(row[('Log Loss', 'mean')]):
            print("  Log Loss: {:.4f} ± {:.4f}".format(row[('Log Loss', 'mean')], row[('Log Loss', 'std')]))
        print("  Total Time: {:.4f} ± {:.4f} seconds".format(row[('Total Time (s)', 'mean')], row[('Total Time (s)', 'std')]))
    
print("\n" + "="*80)
print("REPORT GENERATED SUCCESSFULLY")
print("Output files can be found in the current directory.")
print("="*80)

def main():
    """Main function to run the evaluation."""
    # Load and preprocess data
    file_path = input("Enter the path to your dataset CSV file: ")
    X, y, unique_classes = load_and_preprocess_data(file_path)
    
    # Store results
    all_metrics = []
    class_metrics = []
    timing_info = []
    
    # Initialize classifiers
    classifiers = {
        'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE)
    }
    
    # Perform k-fold cross validation
    kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X)):
        print(f"\nProcessing fold {fold_idx+1}/{K_FOLDS}...")
        
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Apply FLD to reduce dimensionality
        fld_start = time.time()
        lda, X_train_lda, X_test_lda = perform_fld(X_train, y_train, X_test)
        fld_time = time.time() - fld_start
        
        # Evaluate each classifier on the FLD-transformed data
        for clf_name, clf in classifiers.items():
            print(f"Evaluating {clf_name} on fold {fold_idx+1}...")
            
            # Train the classifier
            train_start = time.time()
            clf.fit(X_train_lda, y_train)
            train_time = time.time() - train_start
            
            # Make predictions
            test_start = time.time()
            y_pred = clf.predict(X_test_lda)
            test_time = time.time() - test_start
            
            # Get probability scores if available
            try:
                y_proba = clf.predict_proba(X_test_lda)
                has_proba = True
            except AttributeError:
                y_proba = None
                has_proba = False
            
            # Record timing information
            current_timing = {
                'Classifier': clf_name,
                'Fold': fold_idx,
                'FLD Time (s)': fld_time,
                'Training Time (s)': train_time,
                'Testing Time (s)': test_time,
                'Total Time (s)': fld_time + train_time + test_time
            }
            timing_info.append(current_timing)
            
            # Calculate confusion matrix
            cm = confusion_matrix(y_test, y_pred)
            cm_metrics = confusion_matrix_metrics(cm, unique_classes)
            
            # Plot confusion matrix
            plot_confusion_matrix(cm, unique_classes, clf_name, fold_idx)
            
            # Calculate overall metrics
            metrics = {
                'Classifier': clf_name,
                'Fold': fold_idx,
                'Accuracy': accuracy_score(y_test, y_pred),
                'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
                'MCC': matthews_corrcoef(y_test, y_pred),
                'Kappa': cohen_kappa_score(y_test, y_pred),
                'FLD Time (s)': fld_time,
                'Training Time (s)': train_time,
                'Testing Time (s)': test_time,
                'Total Time (s)': fld_time + train_time + test_time
            }
            
            # Add probability-based metrics if available
            if has_proba:
                metrics['ROC AUC'] = calculate_roc_auc_safely(y_test, y_proba)
                metrics['Log Loss'] = calculate_log_loss_safely(y_test, y_proba, unique_classes)
                
                # Plot ROC curve
                plot_roc_curve(y_test, y_proba, unique_classes, clf_name, fold_idx)
                
                # Plot PR curve
                plot_pr_curve(y_test, y_proba, unique_classes, clf_name, fold_idx)
                
                # Plot calibration curve
                plot_calibration_curve(y_test, y_proba, clf_name, fold_idx)
            else:
                metrics['ROC AUC'] = np.nan
                metrics['Log Loss'] = np.nan
            
            all_metrics.append(metrics)
            
            # Calculate per-class metrics
            for class_label in unique_classes:
                class_metric = calculate_class_metrics(
                    y_test, y_pred, class_label, cm_metrics, fold_idx, clf_name, 
                    {'fld_time': fld_time, 'train_time': train_time, 'test_time': test_time}
                )
                class_metrics.append(class_metric)
            
            # Save detailed classification report
            save_classification_report(y_test, y_pred, clf_name, fold_idx)
            
            # Plot FLD projection if applicable
            plot_fld_projection(X_train_lda, y_train, clf_name, fold_idx)
    
    # Convert results to DataFrame
    all_metrics_df = pd.DataFrame(all_metrics)
    class_metrics_df = pd.DataFrame(class_metrics)
    timing_df = pd.DataFrame(timing_info)
    
    # Save results to CSV
    all_metrics_df.to_csv("fld_overall_metrics.csv", index=False)
    class_metrics_df.to_csv("fld_class_metrics.csv", index=False)
    timing_df.to_csv("fld_timing_info.csv", index=False)
    
    # Generate summary visualizations
    generate_summary_visualizations(all_metrics_df, class_metrics_df, timing_df)
    
    # Perform statistical analysis
    stat_df = perform_statistical_analysis(all_metrics_df)
    
    # Print final report
    print_final_report(all_metrics_df, timing_df, class_metrics_df)

if __name__ == "__main__":
    main()


REPORT GENERATED SUCCESSFULLY
Output files can be found in the current directory.
